In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations

folder = "../database_cleaned"

In [ ]:
jeux_clean = pd.read_csv(f'{folder}/jeux_clean.csv')
fig = plt.figure(figsize=(10, 5))
jeux_clean["Type"] = jeux_clean["Type"].str.split('|')
jeux_expand = jeux_clean.explode("Type")

In [ ]:
# Categories visualization
def plot_hist_count(data, ax, x_labels_vis = False, y_label=None, title=None, rotation=90, sorted=False):
    """Plot histogram (bars). Data : doit être formé de 2 colonnes : index et values (e.g. à l'aide de group by)"""

    
    ax.clear()
    
    if sorted:
        data.sort_index(inplace=True)

    x_coords = np.arange(stop = data.shape[0])

    bars = ax.bar(x_coords, height=data.values.flatten())

    nan_check = data.index.isna()

    if nan_check.any(): # change NaN bar color to red
            x_labels = data.index
            bars[np.where(nan_check)[0][0]].set_color('r')
            
    if x_labels_vis:
        ax.tick_params(axis='x', labelrotation=rotation)
        ax.set_xticks(x_coords)  # Set the tick positions
        ax.set_xticklabels(x_labels)
        ax.xaxis.set_major_locator(plt.FixedLocator(x_coords))
    else:
        ax.xaxis.set_visible(False)

    if title:
        ax.set_title(title)

    if y_label:
        ax.set_ylabel(y_label)
    return ax


def heatmap_categories(matrix_df, ax, cbarlabel):
    """Plot heatmap (2D = 2 categories) to see most common categories combinations"""
    
    ax.clear()
    heatmap = ax.imshow(matrix_df.to_numpy())
    cbar = ax.figure.colorbar(heatmap, ax=ax)
    cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    labels = matrix_df.columns
    ax.set_xticks(range(len(labels)))  # Set the tick positions
    ax.set_xticklabels(labels)

    ax.set_yticks(range(len(labels)))  # Set the tick positions
    ax.set_yticklabels(labels)
    
    ax.tick_params(axis='x', labelrotation=90)

    ax.xaxis.set_major_locator(plt.FixedLocator(np.arange(len(labels))))
    ax.yaxis.set_major_locator(plt.FixedLocator(np.arange(len(labels))))


In [ ]:
categories_data = jeux_expand[["Game id", "Type"]].groupby("Type", dropna=False, as_index=True).count()
categories_popular = categories_data.sort_values("Game id", ascending=False)


In [ ]:
fig.clear()
ax = plot_hist_count(data=categories_popular.head(20), ax=fig.add_subplot(111),
                       x_labels_vis=True, y_label="Number of games", title="Games per category")
fig

In [ ]:
# Categories histogram (all 183 categories)
fig.clear()
ax = plot_hist_count(data=categories_popular["Game id"], ax=fig.add_subplot(111),
                       x_labels_vis=False, y_label="Number of games", title="Games per category")
fig


In [ ]:
# Create matrix (= dataframe) for heatmap (20 most common categories)
top_categories = categories_popular[~categories_popular.index.isna()].head(20).index # top N categories
pair_categories = Counter([pair for cats in jeux_clean["Type"][~jeux_clean["Type"].isna()] for pair in combinations(sorted(cats), 2)])

nb_categories = top_categories.size
heatmap_matrix = pd.DataFrame(data=0, index=top_categories, columns=top_categories)

for (cat1, cat2), count in pair_categories.items():
    if cat1 in heatmap_matrix and cat2 in heatmap_matrix.columns:
        heatmap_matrix.loc[cat1, cat2] += count
    if cat2 in heatmap_matrix and cat1 in heatmap_matrix.columns:
        heatmap_matrix.loc[cat2, cat1] += count
heatmap_matrix

In [ ]:
fig.clear()
ax = heatmap_categories(heatmap_matrix, fig.add_subplot(111), "Nombre de jeux")
fig

In [ ]:
fig.clear()